# 🔄 03 Dataset Pipeline
Build a high-performance `tf.data` pipeline.

### 🎯 Goals:
1. Load processed configuration
2. Perform stratified split
3. Apply `AUTOTUNE` optimizations (Prefetching, Caching)
4. Integrated Data Augmentation

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/DL/'
PROCESSED_PATH = os.path.join(BASE_PATH, 'processed_data/')

import json
import tensorflow as tf
from tensorflow.keras import layers

# Load configuration
with open(os.path.join(PROCESSED_PATH, 'preprocessing_config.json'), 'r') as f:
    config = json.load(f)

DATASET_PATH = config['dataset_path']
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
SEED = 42

# 1. Create Datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.3,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.3,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_batches = tf.data.experimental.cardinality(val_test_ds) // 2
val_ds = val_test_ds.take(val_batches)
test_ds = val_test_ds.skip(val_batches)

class_names = train_ds.class_names
print(f"✅ Classes: {class_names}")

### ⚡ Pipeline Optimizations

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def prepare_pipeline(ds, augment=False):
    ds = ds.map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUTOTUNE)
    if augment:
        data_augmentation = tf.keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.2),
            layers.RandomZoom(0.2),
        ])
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds = prepare_pipeline(train_ds, augment=True)
val_ds = prepare_pipeline(val_ds)
test_ds = prepare_pipeline(test_ds)

print("✅ tf.data pipelines configured.")